In [1]:
%pip install datasets pandas pillow pyarrow

  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 6.6 MB/s  0:00:00
Using cached fsspec-2026.7.0-py3-none-any.whl (206 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 8.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 8.7 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.9/552.9 kB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [datasets]/20 [datasets]ce-hub]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

# Load first 20 products
print("Loading product dataset...")

dataset = load_dataset(
    "ashraq/fashion-product-images-small",
    split="train[:20]"
)

print(f"Loaded {len(dataset)} products!")

# Convert metadata to DataFrame
products_df = dataset.to_pandas()

# Display available columns
print(products_df.columns.tolist())

# Preview products
display(products_df.head())

/opt/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading product dataset...


Generating train split: 100%|██████████| 44072/44072 [00:00<00:00, 185846.39 examples/s]

Loaded 20 products!
['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...


In [3]:
# Create folder
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

# Save images
for i, product in enumerate(dataset):

    image = product["image"]

    image_path = images_dir / f"product_{i+1}.jpg"

    # Convert to RGB if necessary
    image = image.convert("RGB")

    # Save image
    image.save(image_path)

print("All product images saved!")

All product images saved!


In [4]:
import random

random.seed(42)

products = []

for i, product in enumerate(dataset):

    products.append({
        "id": product["id"],
        "name": product["productDisplayName"],
        "price": round(random.uniform(19.99, 199.99), 2),
        "category": product["masterCategory"],
        "subcategory": product["subCategory"],
        "color": product["baseColour"],
        "image_path": f"product_images/product_{i+1}.jpg"
    })

products_df = pd.DataFrame(products)

display(products_df.head())

,id,name,price,category,subcategory,color,image_path
0,15970,Turtle Check Men Navy Blue Shirt,135.09,Apparel,Topwear,Navy Blue,product_images/product_1.jpg
1,39386,Peter England Men Party Blue Jeans,24.49,Apparel,Bottomwear,Blue,product_images/product_2.jpg
2,59263,Titan Women Silver Watch,69.50,Accessories,Watches,Silver,product_images/product_3.jpg
3,21379,Manchester United Men Solid Black Track Pants,60.17,Apparel,Bottomwear,Black,product_images/product_4.jpg
4,53759,Puma Men Grey T-shirt,152.55,Apparel,Topwear,Grey,product_images/product_5.jpg


In [5]:
products_df.to_csv(
    "products.csv",
    index=False
)

print("Product metadata saved successfully!")

Product metadata saved successfully!


In [6]:
import base64

def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    
    return encoded

In [7]:
sample_path = products_df.iloc[0]["image_path"]

encoded_image = encode_image_to_base64(sample_path)

print(f"Image path: {sample_path}")
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Image path: product_images/product_1.jpg
Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


In [8]:
from pathlib import Path

# Decode Base64 back to binary
decoded_image = base64.b64decode(encoded_image)

# Read original image
original_image = Path(sample_path).read_bytes()

# Compare
print("Encoding successful:", decoded_image == original_image)

Encoding successful: True


In [9]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a structured prompt for AI-generated product listings.
    """

    prompt = f"""
You are an expert e-commerce copywriter.

Analyze the provided product image and metadata to create
a professional, engaging, and SEO-friendly product listing.

PRODUCT INFORMATION:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Information: {additional_info}' if additional_info else ''}

REQUIREMENTS:

1. PRODUCT TITLE
- Catchy and SEO-friendly
- Maximum 60 characters
- Include the product type and important attributes

2. PRODUCT DESCRIPTION
- 150-200 words
- Highlight key features and benefits
- Use professional and persuasive language
- Describe visible design elements and colors
- Avoid unsupported claims

3. KEY FEATURES
- Provide 5-7 bullet points
- Focus on relevant product characteristics
- Use clear and concise language

4. SEO KEYWORDS
- Provide 10-15 relevant keywords
- Include product type, category, and relevant attributes

IMPORTANT RULES:
- Use the product image as the primary visual reference.
- Do not invent materials, specifications, or technical features.
- Only mention attributes supported by the image or metadata.
- Return valid JSON only, without markdown formatting.

OUTPUT FORMAT:

{{
    "title": "Product title here",
    "description": "Full product description here",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "keyword1, keyword2, keyword3"
}}
"""

    return prompt

In [10]:
# Select the first product
product = products_df.iloc[0]

# Generate prompt
test_prompt = create_product_listing_prompt(
    product_name=product["name"],
    price=product["price"],
    category=product["category"],
    additional_info=f"""
    Subcategory: {product['subcategory']}
    Color: {product['color']}
    """
)

# Display prompt
print("=" * 50)
print("PRODUCT LISTING PROMPT")
print("=" * 50)
print(test_prompt)

PRODUCT LISTING PROMPT

You are an expert e-commerce copywriter.

Analyze the provided product image and metadata to create
a professional, engaging, and SEO-friendly product listing.

PRODUCT INFORMATION:
- Name: Turtle Check Men Navy Blue Shirt
- Price: $135.09
- Category: Apparel
- Additional Information: 
    Subcategory: Topwear
    Color: Navy Blue
    

REQUIREMENTS:

1. PRODUCT TITLE
- Catchy and SEO-friendly
- Maximum 60 characters
- Include the product type and important attributes

2. PRODUCT DESCRIPTION
- 150-200 words
- Highlight key features and benefits
- Use professional and persuasive language
- Describe visible design elements and colors
- Avoid unsupported claims

3. KEY FEATURES
- Provide 5-7 bullet points
- Focus on relevant product characteristics
- Use clear and concise language

4. SEO KEYWORDS
- Provide 10-15 relevant keywords
- Include product type, category, and relevant attributes

IMPORTANT RULES:
- Use the product image as the primary visual reference.
- D

In [11]:
# Verify required information
assert product["name"] in test_prompt
assert product["category"] in test_prompt
assert "PRODUCT TITLE" in test_prompt
assert "PRODUCT DESCRIPTION" in test_prompt
assert "KEY FEATURES" in test_prompt
assert "SEO KEYWORDS" in test_prompt
assert "JSON" in test_prompt

print("✓ Prompt validation successful!")

✓ Prompt validation successful!


In [12]:
%pip install --upgrade openai

Note: you may need to restart the kernel to use updated packages.


In [15]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv(".env")

# Initialize OpenAI client
client = OpenAI()

print("OpenAI client initialized successfully!")

OpenAI client initialized successfully!


In [17]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": test_prompt
                },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{encoded_image}"
                }
            ]
        }
    ],
    store=False
)

print("API request completed!")
print("Response ID:", response.id)
print("Response status:", response.status)

API request completed!
Response ID: resp_0ee29b8eaee6cd4b016ac7b77eafd887d2a0e1b538bf52dffc
Response status: completed


In [18]:
print(response.output_text)

{
    "title": "Navy Blue Turtle Check Men's Shirt – Stylish & Modern",
    "description": "Elevate your wardrobe with the Turtle Check Men's Navy Blue Shirt, a perfect blend of comfort and sophistication. Designed with a timeless check pattern, this shirt is ideal for both casual outings and formal occasions. The rich navy blue color exudes elegance, making it a versatile choice for various ensembles. Featuring a tailored fit, this shirt provides an effortless look while ensuring all-day comfort. The carefully crafted short sleeves and neat collar add a contemporary touch, appealing to modern fashion sensibilities. Pair it with jeans or chinos for an effortlessly stylish outfit that stands out. Invest in this wardrobe staple to experience unmatched style and comfort.",
    "features": [
        "Timeless check pattern for a classic look",
        "Rich navy blue color for versatility",
        "Tailored fit for a modern silhouette",
        "Short sleeves for comfort in warm weather",

In [19]:
import json

# Convert JSON string to Python dictionary
listing = json.loads(response.output_text)

# Verify the result
print("Data type:", type(listing))
print("Available fields:", list(listing.keys()))

print("\nProduct Title:")
print(listing["title"])

print("\nNumber of Features:")
print(len(listing["features"]))

Data type: <class 'dict'>
Available fields: ['title', 'description', 'features', 'keywords']

Product Title:
Navy Blue Turtle Check Men's Shirt – Stylish & Modern

Number of Features:
7


In [20]:
# Initialize results with the already processed product

results = [
    {
        "product_id": product["id"],
        "product_name": product["name"],
        "price": product["price"],
        "category": product["category"],
        "listing": listing,
        "status": "success"
    }
]

print("Results initialized!")
print("Products already processed:", len(results))

Results initialized!
Products already processed: 1


In [21]:
import json
import time

# Process products 2, 3, and 4
for i in range(1, 4):

    product = products_df.iloc[i]

    print(f"\nProcessing product {i+1}: {product['name']}")

    try:
        # 1. Create prompt
        prompt = create_product_listing_prompt(
            product_name=product["name"],
            price=product["price"],
            category=product["category"],
            additional_info=f"Color: {product['color']}, Subcategory: {product['subcategory']}"
        )

        # 2. Encode image
        encoded_image = encode_image_to_base64(
            product["image_path"]
        )

        # 3. Call OpenAI API
        response = client.responses.create(
            model="gpt-4o-mini",
            input=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": prompt
                        },
                        {
                            "type": "input_image",
                            "image_url": f"data:image/jpeg;base64,{encoded_image}"
                        }
                    ]
                }
            ],
            store=False
        )

        # 4. Parse JSON response
        generated_listing = json.loads(response.output_text)

        # 5. Save successful result
        results.append({
            "product_id": product["id"],
            "product_name": product["name"],
            "price": product["price"],
            "category": product["category"],
            "listing": generated_listing,
            "status": "success"
        })

        print("✓ Successfully processed!")

    except Exception as e:

        # Save failed result
        results.append({
            "product_id": product["id"],
            "product_name": product["name"],
            "price": product["price"],
            "category": product["category"],
            "listing": None,
            "status": "failed",
            "error": str(e)
        })

        print(f"✗ Error: {e}")

    # Small delay between requests
    time.sleep(1)

print("\nBatch processing completed!")


Processing product 2: Peter England Men Party Blue Jeans
✓ Successfully processed!

Processing product 3: Titan Women Silver Watch
✓ Successfully processed!

Processing product 4: Manchester United Men Solid Black Track Pants
✓ Successfully processed!

Batch processing completed!


In [22]:
print("Total results:", len(results))

for item in results:
    print(item["product_name"], "→", item["status"])

Total results: 4
Turtle Check Men Navy Blue Shirt → success
Peter England Men Party Blue Jeans → success
Titan Women Silver Watch → success
Manchester United Men Solid Black Track Pants → success


In [23]:
# Convert nested results into a flat DataFrame

listings_df = pd.json_normalize(results, sep="_")

display(listings_df.head())

,product_id,product_name,price,category,status,listing_title,listing_description,listing_features,listing_keywords
0,15970,Turtle Check Men Navy Blue Shirt,135.09,Apparel,success,Navy Blue Turtle Check Men's Shirt – Stylish &...,Elevate your wardrobe with the Turtle Check Me...,"[Timeless check pattern for a classic look, Ri...","men's shirt, navy blue shirt, turtle check shi..."
1,39386,Peter England Men Party Blue Jeans,24.49,Apparel,success,Peter England Men's Blue Party Jeans - Stylish...,Step out in style with the Peter England Men P...,"[Rich blue color for a sophisticated look, Com...","Peter England jeans, men's blue jeans, party j..."
2,59263,Titan Women Silver Watch,69.50,Accessories,success,Elegant Titan Women Silver Watch - Timeless Style,Discover the exquisite Titan Women Silver Watc...,"[Sleek silver finish for a modern look, Minima...","Titan women silver watch, women watches, silve..."
3,21379,Manchester United Men Solid Black Track Pants,60.17,Apparel,success,Men's Solid Black Manchester United Track Pants,Elevate your casual wear with the Manchester U...,"[Official Manchester United logo, Sleek black ...","Manchester United track pants, men's apparel, ..."


In [24]:
# Save generated listings

listings_df.to_csv(
    "generated_listings.csv",
    index=False
)

print("Generated listings saved successfully!")
print("Total products:", len(listings_df))

Generated listings saved successfully!
Total products: 4


In [25]:
# Read the saved CSV

saved_df = pd.read_csv("generated_listings.csv")

print("Saved products:", len(saved_df))

print("\nProcessing status:")
print(saved_df["status"].value_counts())

print("\nAvailable columns:")
print(saved_df.columns.tolist())

Saved products: 4

Processing status:
status
success    4
Name: count, dtype: int64

Available columns:
['product_id', 'product_name', 'price', 'category', 'status', 'listing_title', 'listing_description', 'listing_features', 'listing_keywords']


In [26]:
# Inspect token usage from the last API response

print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("Total tokens:", response.usage.total_tokens)

Input tokens: 8803
Output tokens: 265
Total tokens: 9068


In [28]:
# GPT-4o-mini pricing per 1 million tokens (USD)
INPUT_PRICE = 0.15
CACHED_INPUT_PRICE = 0.075
OUTPUT_PRICE = 0.60

usage = response.usage

input_tokens = usage.input_tokens
output_tokens = usage.output_tokens

# Check whether any input tokens were cached
cached_tokens = (
    usage.input_tokens_details.cached_tokens
    if usage.input_tokens_details else 0
) or 0

uncached_tokens = input_tokens - cached_tokens

# Calculate cost
input_cost = (
    uncached_tokens * INPUT_PRICE
    + cached_tokens * CACHED_INPUT_PRICE
) / 1_000_000

output_cost = output_tokens * OUTPUT_PRICE / 1_000_000

total_cost = input_cost + output_cost

print(f"Uncached input tokens: {uncached_tokens}")
print(f"Cached input tokens: {cached_tokens}")
print(f"Output tokens: {output_tokens}")

print(f"\nInput cost: ${input_cost:.6f}")
print(f"Output cost: ${output_cost:.6f}")
print(f"Total request cost: ${total_cost:.6f}")

Uncached input tokens: 8803
Cached input tokens: 0
Output tokens: 265

Input cost: $0.001320
Output cost: $0.000159
Total request cost: $0.001479


In [29]:
def estimate_batch_cost(number_of_products, cost_per_request, regeneration_rate=0):
    """
    Estimate API costs for processing multiple products.

    Assumes each regenerated product requires one extra API call.
    """

    initial_cost = number_of_products * cost_per_request

    regeneration_cost = initial_cost * regeneration_rate

    total_estimated_cost = initial_cost + regeneration_cost

    return {
        "products": number_of_products,
        "initial_cost": round(initial_cost, 4),
        "regeneration_cost": round(regeneration_cost, 4),
        "total_cost": round(total_estimated_cost, 4)
    }

In [30]:
scenarios = [100, 1000, 10000, 100000]

for n in scenarios:

    estimate = estimate_batch_cost(
        number_of_products=n,
        cost_per_request=total_cost,
        regeneration_rate=0.10
    )

    print(
        f"{n:,} products → "
        f"${estimate['total_cost']:.2f}"
    )

100 products → $0.16
1,000 products → $1.63
10,000 products → $16.27
100,000 products → $162.74


In [31]:
print("Current prompt length:", len(test_prompt), "characters")
print("Current prompt words:", len(test_prompt.split()))

Current prompt length: 1356 characters
Current prompt words: 192


In [32]:
def create_optimized_prompt(product_name, price, category, additional_info=None):

    return f"""
Create an accurate, SEO-friendly e-commerce listing using
the product image and metadata.

Product: {product_name}
Price: ${price:.2f}
Category: {category}
Additional info: {additional_info or 'None'}

Return valid JSON with:
- title: SEO-friendly, max 60 characters
- description: 150-200 words, persuasive and factual
- features: array of 5-7 concise features
- keywords: string of 10-15 comma-separated SEO keywords

Describe visible attributes accurately.
Do not invent materials, specifications, or unsupported claims.

JSON fields: title, description, features, keywords.
Return JSON only.
"""

In [33]:
optimized_prompt = create_optimized_prompt(
    product_name=products_df.iloc[0]["name"],
    price=products_df.iloc[0]["price"],
    category=products_df.iloc[0]["category"],
    additional_info=(
        f"Color: {products_df.iloc[0]['color']}, "
        f"Subcategory: {products_df.iloc[0]['subcategory']}"
    )
)

original_length = len(test_prompt)
optimized_length = len(optimized_prompt)

reduction = (1 - optimized_length / original_length) * 100

print(f"Original prompt: {original_length} characters")
print(f"Optimized prompt: {optimized_length} characters")
print(f"Length reduction: {reduction:.1f}%")

Original prompt: 1356 characters
Optimized prompt: 618 characters
Length reduction: 54.4%


In [51]:
def test_prompt_version(prompt, image_base64):

    result = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{image_base64}"
                    }
                ]
            }
        ],
        store=False
    )

    input_tokens = result.usage.input_tokens
    output_tokens = result.usage.output_tokens

    cached_tokens = (
        result.usage.input_tokens_details.cached_tokens
        if result.usage.input_tokens_details else 0
    ) or 0

    cost = (
        (input_tokens - cached_tokens) * INPUT_PRICE
        + cached_tokens * CACHED_INPUT_PRICE
        + output_tokens * OUTPUT_PRICE
    ) / 1_000_000

    # Preserve the raw response before parsing
    raw_output = result.output_text

    raw_output = raw_output.strip()

    if raw_output.startswith("```"):
        raw_output = raw_output.split("\n", 1)[1]
        raw_output = raw_output.rsplit("```", 1)[0].strip()

    try:
        listing = json.loads(raw_output)
        json_valid = True
        error = None

    except json.JSONDecodeError as e:
        listing = None
        json_valid = False
        error = str(e)

    return {
        "listing": listing,
        "raw_output": raw_output,
        "json_valid": json_valid,
        "error": error,
        "response_status": result.status,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": result.usage.total_tokens,
        "cost": cost
    }

In [52]:
v2_result = test_prompt_version(optimized_prompt, encoded_image)

print("API status:", v2_result["response_status"])
print("Valid JSON:", v2_result["json_valid"])
print("Total tokens:", v2_result["total_tokens"])
print("Cost:", v2_result["cost"])

print("\nRaw output:")
print(repr(v2_result["raw_output"][:1000]))

print("\nParsing error:", v2_result["error"])

API status: completed
Valid JSON: True
Total tokens: 8900
Cost: 0.00144975

Raw output:
'{\n  "title": "Navy Blue Turtle Check Men’s Shirt - Stylish Topwear",\n  "description": "Elevate your wardrobe with the Turtle Check Men Navy Blue Shirt, designed for the modern man who values style and comfort. This shirt features a sophisticated navy blue check pattern, perfect for both casual and formal occasions. Crafted from high-quality fabric, it ensures breathability and all-day comfort. Pair it with jeans for a relaxed look or dress it up with trousers for a more refined appearance. The timeless design seamlessly transitions from day to night, making it a versatile addition to your collection. With a perfect fit and premium craftsmanship, this shirt promises to be a staple in your closet.",\n  "features": [\n    "Elegant navy blue check pattern",\n    "Suitable for casual and formal wear",\n    "High-quality, breathable fabric",\n    "Comfortable fit for all-day wear",\n    "Versatile desi

In [38]:
raw_output = v2_result["raw_output"].strip()

# Remove Markdown code fences
if raw_output.startswith("```"):
    raw_output = raw_output.split("\n", 1)[1]
    raw_output = raw_output.rsplit("```", 1)[0].strip()

# Parse cleaned JSON
v2_result["listing"] = json.loads(raw_output)
v2_result["json_valid"] = True
v2_result["error"] = None

print("Valid JSON:", v2_result["json_valid"])
print("Listing fields:", list(v2_result["listing"].keys()))

Valid JSON: True
Listing fields: ['title', 'description', 'features', 'keywords']


In [39]:
comparison = pd.DataFrame([
    {
        "version": "V1 - Original",
        "input_tokens": v1_result["input_tokens"],
        "output_tokens": v1_result["output_tokens"],
        "total_tokens": v1_result["total_tokens"],
        "cost_usd": v1_result["cost"]
    },
    {
        "version": "V2 - Optimized",
        "input_tokens": v2_result["input_tokens"],
        "output_tokens": v2_result["output_tokens"],
        "total_tokens": v2_result["total_tokens"],
        "cost_usd": v2_result["cost"]
    }
])

display(comparison)

cost_reduction = (
    1 - v2_result["cost"] / v1_result["cost"]
) * 100

print(f"Cost reduction: {cost_reduction:.2f}%")

,version,input_tokens,output_tokens,total_tokens,cost_usd
0,V1 - Original,8806,255,9061,0.001474
1,V2 - Optimized,8645,279,8924,0.001464


Cost reduction: 0.66%


In [40]:
def score_listing(listing):

    score = 0
    issues = []

    required_fields = [
        "title",
        "description",
        "features",
        "keywords"
    ]

    # 1. Required fields — 20 points
    if all(field in listing for field in required_fields):
        score += 20
    else:
        issues.append("Missing required fields")

    # 2. Title length — 20 points
    title = listing.get("title", "")

    if isinstance(title, str) and 0 < len(title) <= 60:
        score += 20
    else:
        issues.append("Invalid title length")

    # 3. Description length — 25 points
    description = listing.get("description", "")
    word_count = (
        len(description.split())
        if isinstance(description, str) else 0
    )

    if 150 <= word_count <= 200:
        score += 25
    else:
        issues.append(f"Description length: {word_count} words")

    # 4. Features count — 20 points
    features = listing.get("features", [])

    if isinstance(features, list) and 5 <= len(features) <= 7:
        score += 20
    else:
        issues.append("Invalid features count")

    # 5. Keywords count — 15 points
    keywords = listing.get("keywords", "")

    keyword_count = (
        len([k for k in keywords.split(",") if k.strip()])
        if isinstance(keywords, str) else 0
    )

    if 10 <= keyword_count <= 15:
        score += 15
    else:
        issues.append(f"Keywords count: {keyword_count}")

    return {
        "score": score,
        "passed": score >= 80,
        "issues": issues
    }

In [41]:
v1_quality = score_listing(v1_result["listing"])
v2_quality = score_listing(v2_result["listing"])

print("V1 Quality:")
print(v1_quality)

print("\nV2 Quality:")
print(v2_quality)

V1 Quality:
{'score': 75, 'passed': False, 'issues': ['Description length: 100 words']}

V2 Quality:
{'score': 75, 'passed': False, 'issues': ['Description length: 107 words']}


In [42]:
quality_results = []

for version, result in [
    ("V1", v1_result),
    ("V2", v2_result)
]:

    quality = score_listing(result["listing"])

    quality_results.append({
        "version": version,
        "quality_score": quality["score"],
        "status": "APPROVED" if quality["passed"] else "REVIEW_REQUIRED",
        "issues": quality["issues"],
        "api_cost": result["cost"]
    })

quality_df = pd.DataFrame(quality_results)

display(quality_df)

,version,quality_score,status,issues,api_cost
0,V1,75,REVIEW_REQUIRED,[Description length: 100 words],0.001474
1,V2,75,REVIEW_REQUIRED,[Description length: 107 words],0.001464


In [57]:
def regenerate_if_needed(listing, max_retries=1):

    updated_listing = listing.copy()
    total_regeneration_cost = 0
    regeneration_input_tokens = 0
    regeneration_output_tokens = 0
    attempts = 0

    quality = score_listing(updated_listing)

    while not quality["passed"] and attempts < max_retries:

        # Only handle short descriptions in this exercise
        if not any(
            issue.startswith("Description length:")
            for issue in quality["issues"]
        ):
            break

        prompt = f"""
Rewrite the following e-commerce product description.

Product title: {updated_listing['title']}

Current description:
{updated_listing['description']}

Requirements:
- Target 175-185 words. Do not write fewer than 160 words.
- Preserve the original product information.
- Use clear, persuasive e-commerce language.
- Do not invent materials, specifications, or product claims.
- Return only the rewritten description as plain text.
"""

        result = client.responses.create(
            model="gpt-4o-mini",
            input=prompt,
            store=False
        )

        # Track API cost
        usage = result.usage
        regeneration_input_tokens += usage.input_tokens
        regeneration_output_tokens += usage.output_tokens

        cached_tokens = (
            usage.input_tokens_details.cached_tokens
            if usage.input_tokens_details else 0
        ) or 0

        request_cost = (
            (usage.input_tokens - cached_tokens) * INPUT_PRICE
            + cached_tokens * CACHED_INPUT_PRICE
            + usage.output_tokens * OUTPUT_PRICE
        ) / 1_000_000

        total_regeneration_cost += request_cost
        attempts += 1

        # Replace only the description
        if result.output_text.strip():
            updated_listing["description"] = result.output_text.strip()

        # Re-evaluate quality
        quality = score_listing(updated_listing)

    return {
        "listing": updated_listing,
        "quality": quality,
        "attempts": attempts,
        "regeneration_cost": total_regeneration_cost,
        "input_tokens": regeneration_input_tokens,
        "output_tokens": regeneration_output_tokens,
    }

In [60]:
regenerated_v2_final = regenerate_if_needed(
    regenerated_v2["listing"],
    max_retries=1
)

print("Previous quality:", regenerated_v2["quality"]["score"])
print("Final quality:", regenerated_v2_final["quality"]["score"])

print("New description length:",
      len(regenerated_v2_final["listing"]["description"].split()))

print("Remaining issues:", regenerated_v2_final["quality"]["issues"])

print("Additional cost:",
      f"${regenerated_v2_final['regeneration_cost']:.6f}")

Previous quality: 100
Final quality: 100
New description length: 171
Remaining issues: []
Additional cost: $0.000000


In [58]:
def process_product(product):

    # 1. Generate prompt
    prompt = create_optimized_prompt(
        product_name=product["name"],
        price=product["price"],
        category=product["category"],
        additional_info=(
            f"Color: {product['color']}, "
            f"Subcategory: {product['subcategory']}"
        )
    )

    # 2. Encode image
    image_base64 = encode_image_to_base64(product["image_path"])

    # 3. Generate listing and track initial API cost
    generation = test_prompt_version(prompt, image_base64)

    # 4. Validate JSON
    if not generation["json_valid"]:
        return {
            "product_id": product["id"],
            "status": "JSON_ERROR",
            "quality_score": 0,
            "api_cost": generation["cost"],
            "error": generation["error"]
        }

    listing = generation["listing"]

    # 5. Evaluate quality
    initial_quality = score_listing(listing)

    # 6. Regenerate if necessary
    regeneration = regenerate_if_needed(
        listing,
        max_retries=1
    )

    # 7. Calculate total API cost
    total_api_cost = (
        generation["cost"]
        + regeneration["regeneration_cost"]
    )

    # 8. Return complete result
    return {
        "product_id": product["id"],
        "product_name": product["name"],
        "listing": regeneration["listing"],
        "initial_quality_score": initial_quality["score"],
        "final_quality_score": regeneration["quality"]["score"],
        "quality_status": (
            "APPROVED"
            if regeneration["quality"]["passed"]
            else "REVIEW_REQUIRED"
        ),
        "regeneration_attempts": regeneration["attempts"],
        "input_tokens": (
            generation["input_tokens"]
            + regeneration["input_tokens"]
        ),

        "output_tokens": (
            generation["output_tokens"]
            + regeneration["output_tokens"]
        ),

        "total_tokens": (
            generation["total_tokens"]
            + regeneration["input_tokens"]
            + regeneration["output_tokens"]
        ),
        "total_api_cost": total_api_cost
    }

In [55]:
test_product = products_df.iloc[4]

test_result = process_product(test_product)

In [56]:
print("Product:", test_result.get("product_name"))
print("Status:", test_result["status"] if "status" in test_result
      else test_result["quality_status"])

print("Initial quality:", test_result.get("initial_quality_score"))
print("Final quality:", test_result["final_quality_score"]
      if "final_quality_score" in test_result else "N/A")

print("Regeneration attempts:", test_result.get("regeneration_attempts", 0))
print("Total API cost:", f"${test_result['total_api_cost']:.6f}"
      if "total_api_cost" in test_result
      else f"${test_result['api_cost']:.6f}")

Product: Puma Men Grey T-shirt
Status: APPROVED
Initial quality: 75
Final quality: 100
Regeneration attempts: 1
Total API cost: $0.001591


In [61]:
import time

batch_results = []

# Process products 6 to 10
for i in range(5, 10):

    product = products_df.iloc[i]

    print(f"Processing {i+1}/10: {product['name']}")

    try:
        result = process_product(product)
        batch_results.append(result)

        status = result.get(
            "quality_status",
            result.get("status", "UNKNOWN")
        )

        print(f"Status: {status}")

    except Exception as e:
        batch_results.append({
            "product_id": product["id"],
            "product_name": product["name"],
            "status": "PROCESSING_ERROR",
            "error": str(e)
        })

        print(f"Error: {e}")

    time.sleep(1)

print("\nBatch processing completed!")
print("Total products processed:", len(batch_results))

Processing 6/10: Inkfruit Mens Chain Reaction T-shirt
Status: APPROVED
Processing 7/10: Fabindia Men Striped Green Shirt
Status: APPROVED
Processing 8/10: Jealous 21 Women Purple Shirt
Status: APPROVED
Processing 9/10: Puma Men Pack of 3 Socks
Status: APPROVED
Processing 10/10: Skagen Men Black Watch
Status: APPROVED

Batch processing completed!
Total products processed: 5


In [62]:
batch_df = pd.DataFrame(batch_results)

summary_columns = [
    "product_name",
    "initial_quality_score",
    "final_quality_score",
    "quality_status",
    "regeneration_attempts",
    "total_tokens",
    "total_api_cost"
]

display(batch_df.reindex(columns=summary_columns))

,product_name,initial_quality_score,final_quality_score,quality_status,regeneration_attempts,total_tokens,total_api_cost
0,Inkfruit Mens Chain Reaction T-shirt,75,100,APPROVED,1,9291,0.001593
1,Fabindia Men Striped Green Shirt,75,100,APPROVED,1,9329,0.001609
2,Jealous 21 Women Purple Shirt,75,100,APPROVED,1,9345,0.001618
3,Puma Men Pack of 3 Socks,75,100,APPROVED,1,9338,0.001611
4,Skagen Men Black Watch,75,100,APPROVED,1,9348,0.001615


In [63]:
approved = (batch_df["quality_status"] == "APPROVED").sum()

total_cost = batch_df["total_api_cost"].sum()

total_tokens = batch_df["total_tokens"].sum()

print(f"Products processed: {len(batch_df)}")
print(f"Approved listings: {approved}")
print(f"Approval rate: {approved / len(batch_df) * 100:.1f}%")

print(f"\nTotal tokens: {total_tokens:,}")
print(f"Total API cost: ${total_cost:.6f}")
print(f"Average cost per product: ${total_cost / len(batch_df):.6f}")

Products processed: 5
Approved listings: 5
Approval rate: 100.0%

Total tokens: 46,651
Total API cost: $0.008048
Average cost per product: $0.001610


In [67]:
import json
import numpy as np

def json_converter(obj):
    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


with open("final_batch_results.json", "w", encoding="utf-8") as f:
    json.dump(
        batch_results,
        f,
        indent=2,
        ensure_ascii=False,
        default=json_converter
    )

print("JSON file saved successfully!")

JSON file saved successfully!


In [68]:
with open("final_batch_results.json", "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print("Valid JSON: Yes")
print("Saved products:", len(saved_results))
print("First product:", saved_results[0]["product_name"])

Valid JSON: Yes
Saved products: 5
First product: Inkfruit Mens Chain Reaction T-shirt


In [65]:
final_df = pd.json_normalize(batch_results, sep="_")

final_df.to_csv("final_batch_results.csv", index=False)

print("CSV file saved successfully!")
print("Total records:", len(final_df))

CSV file saved successfully!
Total records: 5


In [66]:
import os

for filename in ["final_batch_results.json", "final_batch_results.csv"]:
    print(
        filename,
        "→",
        "Saved" if os.path.exists(filename) else "Missing"
    )

final_batch_results.json → Saved
final_batch_results.csv → Saved


In [69]:
# Load the original four products
original_df = pd.read_csv("generated_listings.csv")

# Convert product 5 into a DataFrame
product_5_df = pd.json_normalize([test_result], sep="_")

# Convert products 6–10 into a DataFrame
batch_final_df = pd.json_normalize(batch_results, sep="_")

# Combine all products
all_products_df = pd.concat(
    [original_df, product_5_df, batch_final_df],
    ignore_index=True
)

print("Total products:", len(all_products_df))
print("Unique product IDs:", all_products_df["product_id"].nunique())

display(all_products_df[["product_id", "product_name"]])

Total products: 10
Unique product IDs: 10


,product_id,product_name
0,15970,Turtle Check Men Navy Blue Shirt
1,39386,Peter England Men Party Blue Jeans
2,59263,Titan Women Silver Watch
3,21379,Manchester United Men Solid Black Track Pants
4,53759,Puma Men Grey T-shirt
5,1855,Inkfruit Mens Chain Reaction T-shirt
6,30805,Fabindia Men Striped Green Shirt
7,26960,Jealous 21 Women Purple Shirt
8,29114,Puma Men Pack of 3 Socks
9,30039,Skagen Men Black Watch


In [70]:
final_columns = [
    "product_id",
    "product_name",
    "listing_title",
    "listing_description",
    "listing_features",
    "listing_keywords",
    "initial_quality_score",
    "final_quality_score",
    "quality_status",
    "regeneration_attempts",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "total_api_cost"
]

standardized_df = all_products_df.reindex(columns=final_columns)

display(standardized_df.head())

print("Products:", len(standardized_df))
print("Columns:", len(standardized_df.columns))

,product_id,product_name,listing_title,listing_description,listing_features,listing_keywords,initial_quality_score,final_quality_score,quality_status,regeneration_attempts,input_tokens,output_tokens,total_tokens,total_api_cost
0,15970,Turtle Check Men Navy Blue Shirt,Navy Blue Turtle Check Men's Shirt – Stylish &...,Elevate your wardrobe with the Turtle Check Me...,"['Timeless check pattern for a classic look', ...","men's shirt, navy blue shirt, turtle check shi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,39386,Peter England Men Party Blue Jeans,Peter England Men's Blue Party Jeans - Stylish...,Step out in style with the Peter England Men P...,"['Rich blue color for a sophisticated look', '...","Peter England jeans, men's blue jeans, party j...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,59263,Titan Women Silver Watch,Elegant Titan Women Silver Watch - Timeless Style,Discover the exquisite Titan Women Silver Watc...,"['Sleek silver finish for a modern look', 'Min...","Titan women silver watch, women watches, silve...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21379,Manchester United Men Solid Black Track Pants,Men's Solid Black Manchester United Track Pants,Elevate your casual wear with the Manchester U...,"['Official Manchester United logo', 'Sleek bla...","Manchester United track pants, men's apparel, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,53759,Puma Men Grey T-shirt,Puma Men Grey T-Shirt - Stylish Topwear,Discover the perfect blend of comfort and styl...,"[Iconic Puma logo for a sporty look, Premium q...","Puma t-shirt, men's grey top, casual apparel, ...",75.0,100.0,APPROVED,1.0,8643.0,242.0,8885.0,0.001591


Products: 10
Columns: 14


In [71]:
standardized_df.to_csv(
    "ai_product_listings_final.csv",
    index=False
)

print("Final dataset saved successfully!")

Final dataset saved successfully!


In [72]:
verified_df = pd.read_csv("ai_product_listings_final.csv")

print("Total products:", len(verified_df))
print("Unique products:", verified_df["product_id"].nunique())
print("Total columns:", len(verified_df.columns))

print("\nMissing listing titles:",
      verified_df["listing_title"].isna().sum())

print("Missing descriptions:",
      verified_df["listing_description"].isna().sum())

Total products: 10
Unique products: 10
Total columns: 14

Missing listing titles: 0
Missing descriptions: 0
